In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "InsurancePlan"
DATA_PATH = "age_insurance (1)(1).csv"


# --------------------------------------------------------------------------
# 1. Load + preprocess
# --------------------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

# Display dataset
print("Sample of the dataset:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nClass balance:")
print(df[TARGET_COL].value_counts())


# Target
y = df[TARGET_COL].astype(int)

# Features
X = df.drop(columns=[TARGET_COL])


# Find categorical and numerical columns
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()


# Label-encode categorical columns
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le


# Impute missing numerical values
imputer = SimpleImputer(strategy="median")

X[num_cols] = imputer.fit_transform(X[num_cols])


# Scale numerical columns
scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(X[num_cols])


# --------------------------------------------------------------------------
# 2. Train / validation / test split
# --------------------------------------------------------------------------

VAL_SIZE = 0.15
TEST_SIZE = 0.15


# First split test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)


# Then split validation data
val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)


print(
    f"\nTrain: {X_train.shape} | "
    f"Val: {X_val.shape} | "
    f"Test: {X_test.shape}"
)


# --------------------------------------------------------------------------
# 3. Benchmark function
# --------------------------------------------------------------------------

def boosting_benchmark(X_train, y_train, X_val, y_val):

    results = []


    # ---- Model 1: AdaBoost -----------------------------------------------

    ada_base = DecisionTreeClassifier(
        max_depth=2,
        random_state=RANDOM_STATE
    )


    ada = AdaBoostClassifier(
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    )


    t0 = time.time()

    ada.fit(X_train, y_train)

    ada_fit_time = time.time() - t0


    ada_val_pred = ada.predict(X_val)


    results.append({
        "model": "AdaBoost",
        "val_accuracy": accuracy_score(y_val, ada_val_pred),
        "val_f1": f1_score(
            y_val,
            ada_val_pred,
            average="weighted"
        ),
        "best_n_estimators": ada.n_estimators,
        "fit_time_sec": round(ada_fit_time, 2),
    })


    # ---- Model 2: XGBoost -----------------------------------------------

    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


    t0 = time.time()

    xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    xgb_fit_time = time.time() - t0


    xgb_val_pred = xgb.predict(X_val)


    results.append({
        "model": "XGBoost",
        "val_accuracy": accuracy_score(y_val, xgb_val_pred),
        "val_f1": f1_score(
            y_val,
            xgb_val_pred,
            average="weighted"
        ),
        "best_n_estimators": xgb.best_iteration + 1,
        "fit_time_sec": round(xgb_fit_time, 2),
    })


    # Create leaderboard
    results_df = pd.DataFrame(results).sort_values(
        "val_accuracy",
        ascending=False
    ).reset_index(drop=True)


    return results_df


# --------------------------------------------------------------------------
# 4. Run
# --------------------------------------------------------------------------

leaderboard = boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
)


print("\nValidation leaderboard (sorted by val_accuracy):")

print(
    leaderboard.to_string(index=False)
)


# Save results

leaderboard.to_csv(
    "boosting_benchmark_results.csv",
    index=False
)


print(
    "\nSaved results to boosting_benchmark_results.csv"
)

Sample of the dataset:
   Age  Insurance  InsurancePlan
0   56          0              0
1   69          1              2
2   46          1              2
3   32          0              0
4   60          1              2

Dataset shape:
(300, 3)

Class balance:
InsurancePlan
0    129
2     97
1     74
Name: count, dtype: int64

Train: (210, 2) | Val: (45, 2) | Test: (45, 2)

Validation leaderboard (sorted by val_accuracy):
   model  val_accuracy   val_f1  best_n_estimators  fit_time_sec
AdaBoost      0.822222 0.812927                200          0.27
 XGBoost      0.822222 0.812927                310          0.15

Saved results to boosting_benchmark_results.csv
